# Dataset split -- diagnostic report

Thin driver only: downloads the dataset, clones the repo, and calls its
(fully dataset-agnostic, unit-tested) `data.coco_split` functions. No pooling,
redistribution, or manifest logic lives in this notebook -- it all lives in the repo.

## 1. Download the dataset (the one genuinely Roboflow-specific step)

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("efe-tatar").project("yolo-to-coco-traffic-signs-pmo2q")
dataset = project.version(1).download("coco")

datasetDirectory = dataset.location
print(datasetDirectory)

## 2. Clone the repo and import its functions

In [ ]:
import os

repoDirectory = "/content/yolo-edge-quantization"
if not os.path.isdir(repoDirectory):
    !git clone -q https://github.com/efe-tatar/yolo-edge-quantization.git {repoDirectory}
else:
    !git -C {repoDirectory} pull -q

import sys
sys.path.insert(0, repoDirectory)

from data.coco_split import pool_boxes, class_photo_counts, redistribute, build_manifest

## 3. Load the per-split COCO json and pool every box

In [ ]:
import json

splitNames = ["train", "valid", "test"]

cocoBySplit = {}
for splitName in splitNames:
    with open(os.path.join(datasetDirectory, splitName, "_annotations.coco.json")) as jsonFile:
        cocoBySplit[splitName] = json.load(jsonFile)

boxes = pool_boxes(cocoBySplit)
print(len(boxes), "boxes pooled across", sum(len(cocoBySplit[s]["images"]) for s in splitNames), "images")

## 4. Per-class counts -- boxes and distinct photos

In [ ]:
classNames, boxesPerClass, imagesPerClass = class_photo_counts(boxes)
sortedClasses = sorted(classNames, key=boxesPerClass.get, reverse=True)

print("class".ljust(26), "boxes".rjust(6), "photos".rjust(7))
for className in reversed(sortedClasses):
    print(className.ljust(26), str(boxesPerClass[className]).rjust(6),
          str(len(imagesPerClass[className])).rjust(7))

averageBoxes = len(boxes) / len(classNames)
print("
average boxes per class:", round(averageBoxes))
print("classes with fewer than 4 distinct photos:")
for className in reversed(sortedClasses):
    if len(imagesPerClass[className]) < 4:
        print(" ", className, "-", len(imagesPerClass[className]), "photos,", boxesPerClass[className], "boxes")

## 5. Redistribute -- rarest class first, photo-level, 70/15/15

In [ ]:
setOfPhoto, trainOnlyClasses = redistribute(boxes, min_photos=4, seed=42)

print(len(trainOnlyClasses), "classes kept but train-only (< 4 photos):")
print(", ".join(trainOnlyClasses))

## 6. Final table + sanity check

In [ ]:
newBoxesPerSet = {className: {"train": 0, "valid": 0, "test": 0} for className in classNames}
for box in boxes:
    setName = setOfPhoto[(box["split"], box["imageId"])]
    newBoxesPerSet[box["class"]][setName] += 1

print("class".ljust(26), "train".rjust(6), "valid".rjust(6), "test".rjust(5), "total".rjust(6))
for className in reversed(sortedClasses):
    counts = newBoxesPerSet[className]
    tag = " (train-only)" if className in trainOnlyClasses else ""
    print(className.ljust(26), str(counts["train"]).rjust(6), str(counts["valid"]).rjust(6),
          str(counts["test"]).rjust(5), str(boxesPerClass[className]).rjust(6), tag)

photosPerSet = {"train": 0, "valid": 0, "test": 0}
for photo, setName in setOfPhoto.items():
    photosPerSet[setName] += 1
totalPhotos = sum(photosPerSet.values())
for setName in ["train", "valid", "test"]:
    print(setName, ":", photosPerSet[setName], "images (",
          round(100 * photosPerSet[setName] / totalPhotos), "%)")

print("
sanity check -- splittable classes that still ended up with zero valid or test boxes:")
for className in reversed(sortedClasses):
    if className in trainOnlyClasses:
        continue
    counts = newBoxesPerSet[className]
    if counts["valid"] == 0 or counts["test"] == 0:
        print("  WARNING:", className, counts)

## 7. Build the manifest (ready for `DetectionDataset`)

In [ ]:
classToIndex = {className: i for i, className in enumerate(sorted(classNames))}

manifest = build_manifest(boxes, setOfPhoto, classToIndex, datasetDirectory)

for setName in ["train", "valid", "test"]:
    print(setName, ":", len(manifest[setName]), "images")

print("
example sample:", manifest["train"][0])

## 8. Box-size distribution vs. level thresholds

`train/targets.py`'s `assign_level` routes each box to a scale by size, but the
thresholds are never hardcoded in the repo -- they're a required argument, decided
here, against this dataset's real box sizes. `img_size`/`thresholds` below are just
local variables for this check, not something that lives in the repo.

In [ ]:
imgSize = 320  # the input size we're planning to train at
thresholds = (64, 128)  # candidate cutoffs -- what we're sanity-checking

# boxes are still in each photo's ORIGINAL pixel size at this point (pool_boxes
# doesn't resize anything) -- rescale to imgSize first, same as DetectionDataset does,
# so "size" here means the same thing assign_level will see during training
sizeOfImage = {}
for splitName in splitNames:
    for image in cocoBySplit[splitName]["images"]:
        sizeOfImage[(splitName, image["id"])] = (image["width"], image["height"])

sizes = []
for box in boxes:
    origW, origH = sizeOfImage[(box["split"], box["imageId"])]
    sx, sy = imgSize / origW, imgSize / origH
    sizes.append(max(box["width"] * sx, box["height"] * sy))

sizes.sort()
n = len(sizes)
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"p{p}:", round(sizes[int(n * p / 100)], 1), "px")

countPerLevel = {"stride8 (<=t0)": 0, "stride16 (t0-t1)": 0, "stride32 (>t1)": 0}
for size in sizes:
    if size <= thresholds[0]:
        countPerLevel["stride8 (<=t0)"] += 1
    elif size <= thresholds[1]:
        countPerLevel["stride16 (t0-t1)"] += 1
    else:
        countPerLevel["stride32 (>t1)"] += 1

print()
for level, count in countPerLevel.items():
    print(level, ":", count, f"({round(100 * count / n)}%)")